In [0]:

# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_payments
# Source          : payments.csv
# Target          : procurement.bronze.bronze_payments
# Audit Table     : procurement.audit.duplicate_payments
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw payments master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load Payments master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Payment IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_PAYMENTS)
print(AUDIT_DUPLICATE_PAYMENTS)
print(PAYMENTS_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Payments schema
payments_schema = StructType([
    StructField("payment_id", StringType(), False),
    StructField("payment_date", StringType(), True),
    StructField("invoice_id", StringType(), True),
    StructField("amount_paid", DecimalType(18,2), True),
    StructField("currency", StringType(), True),
     StructField("payment_method", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("days_delayed", StringType(), True)  # Changed to StringType to avoid nulls
])
# Read Payments master data from landing volume
bronze_payments_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(payments_schema)
    .load(PAYMENTS_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_payments_df.count()}")

print("\nSchema:")
bronze_payments_df.printSchema()

print("\nColumns:")
print(bronze_payments_df.columns)

print("\nSampledata:")
display(bronze_payments_df.limit(10))

In [0]:
# Check the NULL and Blank payment_ids
null_blank_payment_id = bronze_payments_df.filter(col("payment_id").isNull() | (trim(col("payment_id")) == ""))

print(f"Total NULL or Blank payment_ids : {null_blank_payment_id.count()}")

display(null_blank_payment_id)

In [0]:
#Check the NULL and Blank payement date
null_blank_payment_date = bronze_payments_df.filter(col("payment_date").isNull() | (trim(col("payment_date")) == ""))

print(f"Total NULL or Blank payment_date : {null_blank_payment_date.count()}")

display(null_blank_payment_date)

In [0]:
# Check the NULL and Blank invoice_ids
null_blank_invoice_id = bronze_payments_df.filter(col("invoice_id").isNull() | (trim(col("invoice_id")) == ""))

print(f"Total NULL or Blank invoice_ids : {null_blank_invoice_id.count()}")

display(null_blank_invoice_id)

In [0]:
#Check the negative budget
negative_amount_paid = bronze_payments_df.filter((col("amount_paid") < 0) | (col("amount_paid").isNull()))

print(f"Total Null and negative amounts : {negative_amount_paid.count()}")

display(negative_amount_paid)

In [0]:
#Check the NULL and Blank currency
null_blank_currency = bronze_payments_df.filter(col("currency").isNull() | (trim(col("currency")) == ""))

print(f"Total NULL or Blank currency : {null_blank_currency.count()}")

display(null_blank_currency)

In [0]:
#Check the NULL and Blank payment method
null_blank_payment_method = bronze_payments_df.filter(col("payment_method").isNull() | (trim(col("payment_method")) == ""))

print(f"Total NULL or Blank payment_method : {null_blank_payment_method.count()}")

display(null_blank_payment_method)

In [0]:
#Check the NULL and Blank payment status
null_blank_payment_status = bronze_payments_df.filter(col("payment_status").isNull() | (trim(col("payment_status")) == ""))

print(f"Total NULL or Blank payment_status : {null_blank_payment_status.count()}")

display(null_blank_payment_status)

In [0]:
# ============================================================
# Identify Duplicate payment IDs
# ============================================================

duplicate_payment_keys = (
    bronze_payments_df
        .groupBy("payment_id")
        .count()
        .filter(col("count") > 1)
        .withColumnRenamed("count", "duplicate_count IDs")
)

display(duplicate_payment_keys)

In [0]:
# ============================================================
# Identify Duplicate payment Records
# Business Rule: Keep the first occurrence of each Payment ID and identify subsequent records as duplicates.
# ============================================================

window_spec = Window.partitionBy("payment_id").orderBy("payment_id")

payment_rank_df = (
    bronze_payments_df
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)
display(payment_rank_df)

In [0]:
# ============================================================
# Retrieve Duplicate  Payment Records
# ============================================================

duplicate_payments = (
    payment_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate Payment Records : {duplicate_payments.count()}")
display(duplicate_payments)


In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

duplicate_payments = (
    duplicate_payments
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Payments"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_payments)

In [0]:
# ============================================================
# Write Duplicate Records to Audit Table
# ============================================================

duplicate_count = duplicate_payments.count()

if duplicate_count > 0:

    write_delta(
        df = duplicate_payments,
         table_name = AUDIT_DUPLICATE_PAYMENTS
    )

    print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_PAYMENTS}")

else:

    print("No duplicate Payments records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_payements_final_df = (
    bronze_payments_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("Payments.csv"))
)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(
    df=bronze_payements_final_df,
    table_name=BRONZE_PAYMENTS
)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_payments = spark.table(BRONZE_PAYMENTS)

print(f"Total Bronze Records : {bronze_payments.count()}")

display(bronze_payments)

In [0]:
# ============================================================
# Bronze payments complete summary
# ============================================================
print("=" * 60)
print("Bronze Payment Load Completed Successfully")
print("=" * 60)

print(f"Landing Records : {bronze_payments_df.count()}")

print(f"Audit Records : {duplicate_payments.count()}")

print(f"Bronze Records : {spark.table(BRONZE_PAYMENTS).count()}")